## Figure 5 — Gaussian-noise feasible sets

This section generates the Gaussian-noise feasible-set figure, including the least-squares and constrained weighted-LASSO estimates across Monte Carlo realizations.

### Functions

| Function | Main inputs | Purpose |
|---|---|---|
| `simulate_siso_n1` | `A, B, C, u, noise_level, noise_model` | Simulate the first-order SISO model with the selected measurement-noise distribution. |
| `build_siso_n1_regressor` | `u, y` | Build the regression matrix and aligned output vector for the first-order observer model. |
| `true_theta_siso_n1` | `A, B, C` | Return the true observer parameter vector for the first-order SISO system. |
| `least_squares` | `Phi, Y` | Compute the ordinary least-squares parameter estimate. |
| `make_adaptive_weights` | `Phi, Y, delta, power` | Construct normalized adaptive weights from an initial least-squares estimate. |
| `weighted_lasso_constrained` | `Phi, Y, epsilon, weights, lambda_l2` | Solve the constrained weighted-LASSO problem. |
| `compute_epsilon_star` | `noise_level, theta_y` | Compute the theoretical residual bound used to define the feasible set. |
| `feasible_ellipse_2d` | `Phi, Y, epsilon, n_points` | Construct the two-dimensional feasible ellipse from the residual constraint. |
| `filter_points_inside_feasible_set` | `Phi, Y, theta_all, epsilon, tol` | Retain only parameter estimates that satisfy the displayed feasible-set constraint. |
| `monte_carlo_estimates` | `A, B, C, N_samples, noise_level, noise_model, num_trials, weight_delta, weight_power, lambda_l2, random_seed, epsilon_star, ellipse_points, feasibility_tolerance` | Run the Monte Carlo study and collect feasible sets and estimator results. |
| `plot_all_feasible_sets_with_estimates` | `ellipses, theta_star, theta_ls_all, theta_lasso_all, epsilon_all` | Plot all feasible ellipses together with the true, LS, and weighted-LASSO parameter estimates. |
| `save_tikz` | `ellipses, theta_star, theta_ls_all, theta_lasso_all, tex_path` | Export the figure data and plotting commands to a TikZ/LaTeX file. |
| `save_all` | `fig, ellipses, theta_star, theta_ls_all, theta_lasso_all` | Save the generated figure and associated export files. |


In [ ]:
# Feasible parameter set for SISO n=1
# plus LS and constrained weighted LASSO estimates
#
# System:
#   x(k+1) = A x(k) + B u(k)
#   y(k)   = C x(k) + e(k)
#
# Deadbeat observer, n=1:
#   theta = [theta_u, theta_y] in R^2
#   theta_star = [C B, A]
#
# Feasible set:
#   Theta_N(eps) = {theta : ||Y - Phi theta||_2 <= eps sqrt(T)}
#
# Constrained weighted lasso:
#   min sum_i w_i |theta_i|
#   s.t. ||Y - Phi theta||_2 <= eps sqrt(T)

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp


# SISO n=1 simulation
def simulate_siso_n1(A, B, C, u, noise_level=0.0, noise_model="gaussian"):
    u = np.asarray(u, dtype=float).reshape(-1)
    N = len(u)

    x = np.zeros(N + 1)
    y = np.zeros(N)

    for k in range(N):
        if noise_model == "gaussian":
            e_k = noise_level * np.random.randn()
        elif noise_model == "uniform":
            e_k = np.random.uniform(-noise_level, noise_level)
        else:
            raise ValueError("noise_model must be 'gaussian' or 'uniform'.")

        y[k] = C * x[k] + e_k
        x[k + 1] = A * x[k] + B * u[k]

    return x, y


def build_siso_n1_regressor(u, y):
    u = np.asarray(u, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)

    Phi = np.column_stack([u[:-1], y[:-1]])
    Y = y[1:]

    return Phi, Y


def true_theta_siso_n1(A, B, C):
    return np.array([C * B, A], dtype=float)


# LS
def least_squares(Phi, Y):
    theta_hat, _, _, _ = np.linalg.lstsq(Phi, Y, rcond=None)
    return theta_hat.reshape(-1)


# Adaptive weights
def make_adaptive_weights(Phi, Y, delta=1e-6, power=1.0):
    theta_init = np.linalg.pinv(Phi) @ Y.reshape(-1, 1)
    theta_init = theta_init.flatten()

    weights = 1.0 / ((np.abs(theta_init) + delta) ** power)
    weights = weights / np.mean(weights)

    return weights


# CONSTRAINED weighted lasso
def weighted_lasso_constrained(Phi, Y, epsilon, weights=None, lambda_l2=0.0):
    Phi = np.asarray(Phi, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)

    T, p = Phi.shape

    if weights is None:
        weights = np.ones(p)

    weights = np.asarray(weights, dtype=float).reshape(-1)

    theta = cp.Variable(p)
    residual = Phi @ theta - Y

    weighted_l1 = cp.norm1(cp.multiply(weights, theta))

    if lambda_l2 > 0:
        objective = cp.Minimize(
            weighted_l1 + 0.5 * lambda_l2 * cp.sum_squares(theta)
        )
    else:
        objective = cp.Minimize(weighted_l1)

    constraints = [
        cp.norm(residual, 2) <= float(epsilon) * np.sqrt(T)
    ]

    prob = cp.Problem(objective, constraints)

    for solver, kwargs in [
        (
            cp.ECOS,
            dict(
                warm_start=True,
                verbose=False,
                max_iters=8000,
                reltol=1e-7,
                abstol=1e-7,
                feastol=1e-7,
            ),
        ),
        (
            cp.SCS,
            dict(
                warm_start=True,
                verbose=False,
                eps=1e-5,
                max_iters=30000,
                alpha=1.8,
                acceleration_lookback=20,
            ),
        ),
    ]:
        try:
            prob.solve(solver=solver, **kwargs)
            if theta.value is not None:
                return theta.value.reshape(-1)
        except Exception:
            continue

    raise ValueError("Constrained weighted lasso failed.")


# Fixed epsilon selection
def compute_epsilon_star(noise_level, theta_y):
    """
    Use the same epsilon for every realization:

        epsilon_star = 1.25 * mu * sqrt(1 + ||theta_y||_2^2).

    Thus, the theoretical threshold is increased by 25%.
    """
    theta_y = np.asarray(theta_y, dtype=float).reshape(-1)

    epsilon_theoretical = float(noise_level) * np.sqrt(
        1.0 + np.linalg.norm(theta_y, 2) ** 2
    )

    return 1.05 * epsilon_theoretical


# Feasible ellipse
def feasible_ellipse_2d(Phi, Y, epsilon, n_points=600):
    T = Phi.shape[0]

    theta_ls = least_squares(Phi, Y)
    residual = Y - Phi @ theta_ls
    residual_norm_sq = float(residual @ residual)

    eps_min = np.sqrt(residual_norm_sq / T)
    rho_sq = float(epsilon**2 * T - residual_norm_sq)

    if rho_sq <= 0:
        raise ValueError(
            f"The feasible set is empty. eps={epsilon:.6g}, eps_min={eps_min:.6g}. "
            "Increase EPSILON_MULT or set EPSILON manually."
        )

    M = Phi.T @ Phi
    eigvals, eigvecs = np.linalg.eigh(M)

    if np.min(eigvals) <= 1e-12:
        raise ValueError("Phi.T @ Phi is nearly singular. Increase N_SAMPLES.")

    M_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

    t = np.linspace(0.0, 2.0 * np.pi, n_points)
    circle = np.vstack([np.cos(t), np.sin(t)])

    ellipse = theta_ls.reshape(2, 1) + np.sqrt(rho_sq) * M_inv_sqrt @ circle

    return {
        "theta_ls": theta_ls,
        "ellipse": ellipse,
        "rho_sq": rho_sq,
        "eps_min": eps_min,
        "residual_norm": np.sqrt(residual_norm_sq),
    }


# Keep only points inside displayed feasible set
def filter_points_inside_feasible_set(Phi, Y, theta_all, epsilon, tol=1e-10):
    Phi = np.asarray(Phi, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    theta_all = np.asarray(theta_all, dtype=float)

    T = Phi.shape[0]
    bound = float(epsilon) * np.sqrt(T)

    keep = []

    for theta in theta_all:
        residual_norm = np.linalg.norm(Y - Phi @ theta, 2)
        keep.append(residual_norm <= bound + tol)

    keep = np.array(keep, dtype=bool)

    return theta_all[keep], keep


# Monte Carlo estimates and one ellipse per feasible realization
def monte_carlo_estimates(
    A,
    B,
    C,
    N_samples,
    noise_level,
    noise_model,
    num_trials,
    weight_delta,
    weight_power,
    lambda_l2,
    random_seed,
    epsilon_star,
    ellipse_points=400,
    feasibility_tolerance=1e-12,
):
    """
    Use the same fixed epsilon_star for every realization and discard
    realizations whose feasible set is empty.

    A realization is retained only if

        eps_min,omega < epsilon_star,

    where eps_min,omega is the normalized LS residual.
    """
    rng_state = np.random.get_state()
    np.random.seed(random_seed)

    theta_star = true_theta_siso_n1(A, B, C)
    epsilon_star = float(epsilon_star)

    theta_ls_kept = []
    theta_lasso_kept = []
    eps_min_kept = []
    eps_true_kept = []
    ellipses = []
    Phi_kept = []
    Y_kept = []
    kept_trial_indices = []
    discarded_trial_indices = []

    for trial in range(num_trials):
        u = np.random.randn(N_samples)

        _, y = simulate_siso_n1(
            A,
            B,
            C,
            u,
            noise_level=noise_level,
            noise_model=noise_model,
        )

        Phi, Y = build_siso_n1_regressor(u, y)
        theta_ls = least_squares(Phi, Y)
        T = Phi.shape[0]

        eps_min = np.linalg.norm(Y - Phi @ theta_ls) / np.sqrt(T)
        eps_true = np.linalg.norm(Y - Phi @ theta_star) / np.sqrt(T)

        tolerance = float(feasibility_tolerance) * max(
            1.0,
            abs(epsilon_star),
            abs(eps_min),
        )

        # At eps_min == epsilon_star, the ellipse collapses to the LS point.
        # Keep only strictly nonempty, nondegenerate ellipses.
        if epsilon_star <= eps_min + tolerance:
            discarded_trial_indices.append(trial)
            continue

        weights = make_adaptive_weights(
            Phi,
            Y,
            delta=weight_delta,
            power=weight_power,
        )

        theta_lasso = weighted_lasso_constrained(
            Phi,
            Y,
            epsilon=epsilon_star,
            weights=weights,
            lambda_l2=lambda_l2,
        )

        ellipse_result = feasible_ellipse_2d(
            Phi,
            Y,
            epsilon_star,
            n_points=ellipse_points,
        )

        theta_ls_kept.append(theta_ls)
        theta_lasso_kept.append(theta_lasso)
        eps_min_kept.append(eps_min)
        eps_true_kept.append(eps_true)
        ellipses.append(ellipse_result["ellipse"])
        Phi_kept.append(Phi)
        Y_kept.append(Y)
        kept_trial_indices.append(trial)

    np.random.set_state(rng_state)

    if not kept_trial_indices:
        raise ValueError(
            "No realization produced a nonempty ellipse with the fixed "
            f"epsilon_star={epsilon_star:.6g}."
        )

    theta_ls_all = np.asarray(theta_ls_kept, dtype=float)
    theta_lasso_all = np.asarray(theta_lasso_kept, dtype=float)
    eps_min_all = np.asarray(eps_min_kept, dtype=float)
    eps_true_all = np.asarray(eps_true_kept, dtype=float)
    epsilon_all = np.full(len(kept_trial_indices), epsilon_star, dtype=float)

    return {
        "theta_ls_all": theta_ls_all,
        "theta_lasso_all": theta_lasso_all,
        "epsilon_star": epsilon_star,
        "epsilon_all": epsilon_all,
        "eps_min_all": eps_min_all,
        "eps_true_all": eps_true_all,
        "ellipses": ellipses,
        "Phi_all": Phi_kept,
        "Y_all": Y_kept,
        "kept_trial_indices": np.asarray(kept_trial_indices, dtype=int),
        "discarded_trial_indices": np.asarray(
            discarded_trial_indices,
            dtype=int,
        ),
        "num_requested": int(num_trials),
        "num_kept": len(kept_trial_indices),
        "num_discarded": len(discarded_trial_indices),
    }


# Plot all realization-specific ellipses and estimates
def plot_all_feasible_sets_with_estimates(
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
    epsilon_all,
):
    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    # Plot one feasible ellipse for every realization.
    for i, ellipse in enumerate(ellipses):
        ax.plot(
            ellipse[0, :],
            ellipse[1, :],
            linewidth=0.8,
            alpha=0.22,
            color="tab:blue",
            label=(
                r"Realization-specific feasible sets"
                if i == 0
                else None
            ),
        )

    ax.scatter(
        theta_ls_all[:, 0],
        theta_ls_all[:, 1],
        s=30,
        alpha=0.70,
        marker="o",
        facecolors="none",
        edgecolors="black",
        linewidths=0.9,
        label=r"LS estimates",
        zorder=4,
    )

    ax.scatter(
        theta_lasso_all[:, 0],
        theta_lasso_all[:, 1],
        s=34,
        alpha=0.82,
        marker="x",
        color="tab:orange",
        linewidths=1.1,
        label=r"Weighted-lasso estimates",
        zorder=5,
    )

    ax.plot(
        theta_star[0],
        theta_star[1],
        marker="*",
        markersize=16,
        markeredgecolor="black",
        linestyle="None",
        color="tab:green",
        label=r"$\theta^\star$",
        zorder=6,
    )

    epsilon_fixed = float(epsilon_all[0])

    ax.set_xlabel(r"$\theta_u$")
    ax.set_ylabel(r"$\theta_y$")
    ax.set_title(
        rf"All {len(ellipses)} realization-specific feasible sets, "
        rf"$\varepsilon^*={epsilon_fixed:.3g}$"
    )
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_aspect("equal", adjustable="datalim")

    ax.legend(
        frameon=True,
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.20),
        ncol=2,
        borderaxespad=0.0,
    )

    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.90])

    return fig, ax


# Manual TikZ export
def save_tikz(
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
    tex_path,
):
    ellipse_blocks = []

    for i, ellipse in enumerate(ellipses):
        ellipse_coords = "\n".join(
            f"({ellipse[0, j]:.12g},{ellipse[1, j]:.12g})"
            for j in range(ellipse.shape[1])
        )

        legend_option = (
            "\n\\addlegendentry{Realization-specific feasible sets}"
            if i == 0
            else ""
        )

        ellipse_blocks.append(
            rf"""\addplot[
    steelblue,
    line width=0.45pt,
    opacity=0.22
] coordinates {{
{ellipse_coords}
}};{legend_option}
"""
        )

    all_ellipse_code = "\n".join(ellipse_blocks)

    ls_coords = "\n".join(
        f"({theta_ls_all[i, 0]:.12g},{theta_ls_all[i, 1]:.12g})"
        for i in range(theta_ls_all.shape[0])
    )

    lasso_coords = "\n".join(
        f"({theta_lasso_all[i, 0]:.12g},{theta_lasso_all[i, 1]:.12g})"
        for i in range(theta_lasso_all.shape[0])
    )

    tex = rf"""% Manually generated PGFPlots file.
% Requires:
% \usepackage{{xcolor}}
% \usepackage{{tikz}}
% \usepackage{{pgfplots}}
% \pgfplotsset{{compat=1.18}}

\begin{{tikzpicture}}

\definecolor{{steelblue}}{{RGB}}{{31,119,180}}
\definecolor{{darkorange}}{{RGB}}{{255,127,14}}
\definecolor{{forestgreen}}{{RGB}}{{44,160,44}}

\begin{{axis}}[
    width=0.88\linewidth,
    height=0.72\linewidth,
    xlabel={{$\theta_u$}},
    ylabel={{$\theta_y$}},
    axis lines=box,
    axis equal image,
    grid=major,
    legend style={{
        fill=white,
        draw=black,
        at={{(0.5,1.18)}},
        anchor=south,
        font=\scriptsize
    }},
    legend columns=2,
    legend cell align={{left}},
    tick align=outside,
    tick pos=left
]

{all_ellipse_code}

\addplot[
    only marks,
    mark=o,
    mark size=1.8pt,
    black,
    fill=white,
    opacity=0.70
] coordinates {{
{ls_coords}
}};
\addlegendentry{{LS estimates}}

\addplot[
    only marks,
    mark=x,
    mark size=2.0pt,
    darkorange,
    opacity=0.82
] coordinates {{
{lasso_coords}
}};
\addlegendentry{{Weighted-lasso estimates}}

\addplot[
    only marks,
    mark=star,
    mark size=3.8pt,
    forestgreen,
    mark options={{draw=black}}
] coordinates {{
({theta_star[0]:.12g},{theta_star[1]:.12g})
}};
\addlegendentry{{$\theta^\star$}}

\end{{axis}}
\end{{tikzpicture}}
"""

    tex_path = Path(tex_path)
    tex_path.parent.mkdir(parents=True, exist_ok=True)
    tex_path.write_text(tex)


def save_all(
    fig,
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
):
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    pdf_path = FIG_DIR / f"{FIG_BASENAME}.pdf"
    png_path = FIG_DIR / f"{FIG_BASENAME}.png"
    tex_path = FIG_DIR / f"{FIG_BASENAME}.tex"

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")

    save_tikz(
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
        tex_path=tex_path,
    )

    print(f"Saved PDF:  {pdf_path}")
    print(f"Saved PNG:  {png_path}")
    print(f"Saved TikZ: {tex_path}")


# Main


### Main experiment

This block sets the numerical parameters, runs the experiment, and saves the corresponding figure outputs.


In [ ]:
def main():
    theta_star = true_theta_siso_n1(A, B, C)
    theta_y = np.array([theta_star[1]], dtype=float)
    epsilon_star = compute_epsilon_star(NOISE_LEVEL, theta_y)

    mc = monte_carlo_estimates(
        A=A,
        B=B,
        C=C,
        N_samples=N_SAMPLES,
        noise_level=NOISE_LEVEL,
        noise_model=NOISE_MODEL,
        num_trials=NUM_TRIALS,
        weight_delta=WEIGHT_DELTA,
        weight_power=WEIGHT_POWER,
        lambda_l2=LAMBDA_L2,
        random_seed=RANDOM_SEED,
        epsilon_star=epsilon_star,
        ellipse_points=ELLIPSE_POINTS,
    )

    theta_ls_all = mc["theta_ls_all"]
    theta_lasso_all = mc["theta_lasso_all"]
    epsilon_all = mc["epsilon_all"]
    ellipses = mc["ellipses"]

    print("theta size:", 2)
    print("theta_star =", theta_star)
    print(f"number of realizations = {NUM_TRIALS}")
    print(f"fixed epsilon_star (25% increased) = {epsilon_star:.6g}")
    print(f"realizations requested = {mc['num_requested']}")
    print(f"realizations kept = {mc['num_kept']}")
    print(f"realizations discarded = {mc['num_discarded']}")
    print(f"epsilon min  = {np.min(epsilon_all):.6g}")
    print(f"epsilon mean = {np.mean(epsilon_all):.6g}")
    print(f"epsilon max  = {np.max(epsilon_all):.6g}")
    print(
        "mean LS error     =",
        np.mean(np.linalg.norm(theta_ls_all - theta_star, axis=1)),
    )
    print(
        "mean LASSO error  =",
        np.mean(np.linalg.norm(theta_lasso_all - theta_star, axis=1)),
    )

    fig, ax = plot_all_feasible_sets_with_estimates(
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
        epsilon_all=epsilon_all,
    )

    save_all(
        fig=fig,
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
    )

    plt.show()

    return {
        "theta_star": theta_star,
        "epsilon_star": mc["epsilon_star"],
        "kept_trial_indices": mc["kept_trial_indices"],
        "discarded_trial_indices": mc["discarded_trial_indices"],
        "num_requested": mc["num_requested"],
        "num_kept": mc["num_kept"],
        "num_discarded": mc["num_discarded"],
        "theta_ls_all": theta_ls_all,
        "theta_lasso_all": theta_lasso_all,
        "epsilon_all": epsilon_all,
        "eps_min_all": mc["eps_min_all"],
        "eps_true_all": mc["eps_true_all"],
        "ellipses": ellipses,
        "Phi_all": mc["Phi_all"],
        "Y_all": mc["Y_all"],
        "fig": fig,
        "ax": ax,
    }


# User choices
RANDOM_SEED = 1

N_SAMPLES = 1000
NUM_TRIALS = 80
ELLIPSE_POINTS = 400

# Noise model:
#   "gaussian": e(k) ~ N(0, NOISE_LEVEL^2)
#   "uniform":  e(k) ~ U[-NOISE_LEVEL, NOISE_LEVEL]
NOISE_MODEL = "gaussian"
NOISE_LEVEL = 0.10

# Use the same fixed epsilon_star for every realization:
#   epsilon_star = 1.25 * NOISE_LEVEL * sqrt(1 + ||theta_y||_2^2).
# Thus, the theoretical threshold is increased by 25%.
# Realizations with eps_min >= epsilon_star are discarded.
# Here theta_y = A = 0.7.

# Constrained weighted-lasso settings
WEIGHT_DELTA = 1e-6
WEIGHT_POWER = 1.0
LAMBDA_L2 = 0.0

FIG_DIR = Path("Figures")
FIG_BASENAME = "all_realization_feasible_sets_gaussian_white_noise"


# SISO n=1 system
A = 0.70
B = 1.00
C = 1.00


if __name__ == "__main__":
    results_gaussian_feasible_siso = main()


## Figure 6 — Uniform-noise feasible sets

This section generates the corresponding feasible-set figure for uniformly distributed measurement noise.

### Functions

| Function | Main inputs | Purpose |
|---|---|---|
| `simulate_siso_n1` | `A, B, C, u, noise_level, noise_model` | Simulate the first-order SISO model with the selected measurement-noise distribution. |
| `build_siso_n1_regressor` | `u, y` | Build the regression matrix and aligned output vector for the first-order observer model. |
| `true_theta_siso_n1` | `A, B, C` | Return the true observer parameter vector for the first-order SISO system. |
| `least_squares` | `Phi, Y` | Compute the ordinary least-squares parameter estimate. |
| `make_adaptive_weights` | `Phi, Y, delta, power` | Construct normalized adaptive weights from an initial least-squares estimate. |
| `weighted_lasso_constrained` | `Phi, Y, epsilon, weights, lambda_l2` | Solve the constrained weighted-LASSO problem. |
| `compute_epsilon_star` | `noise_level, theta_y` | Compute the theoretical residual bound used to define the feasible set. |
| `feasible_ellipse_2d` | `Phi, Y, epsilon, n_points` | Construct the two-dimensional feasible ellipse from the residual constraint. |
| `filter_points_inside_feasible_set` | `Phi, Y, theta_all, epsilon, tol` | Retain only parameter estimates that satisfy the displayed feasible-set constraint. |
| `monte_carlo_estimates` | `A, B, C, N_samples, noise_level, noise_model, num_trials, weight_delta, weight_power, lambda_l2, random_seed, epsilon_star, ellipse_points, feasibility_tolerance` | Run the Monte Carlo study and collect feasible sets and estimator results. |
| `plot_all_feasible_sets_with_estimates` | `ellipses, theta_star, theta_ls_all, theta_lasso_all, epsilon_all` | Plot all feasible ellipses together with the true, LS, and weighted-LASSO parameter estimates. |
| `save_tikz` | `ellipses, theta_star, theta_ls_all, theta_lasso_all, tex_path` | Export the figure data and plotting commands to a TikZ/LaTeX file. |
| `save_all` | `fig, ellipses, theta_star, theta_ls_all, theta_lasso_all` | Save the generated figure and associated export files. |


In [ ]:
# Feasible parameter set for SISO n=1
# plus LS and constrained weighted LASSO estimates
#
# System:
#   x(k+1) = A x(k) + B u(k)
#   y(k)   = C x(k) + e(k)
#
# Deadbeat observer, n=1:
#   theta = [theta_u, theta_y] in R^2
#   theta_star = [C B, A]
#
# Feasible set:
#   Theta_N(eps) = {theta : ||Y - Phi theta||_2 <= eps sqrt(T)}
#
# Constrained weighted lasso:
#   min sum_i w_i |theta_i|
#   s.t. ||Y - Phi theta||_2 <= eps sqrt(T)

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp


# SISO n=1 simulation
def simulate_siso_n1(A, B, C, u, noise_level=0.0, noise_model="gaussian"):
    u = np.asarray(u, dtype=float).reshape(-1)
    N = len(u)

    x = np.zeros(N + 1)
    y = np.zeros(N)

    for k in range(N):
        if noise_model == "gaussian":
            e_k = noise_level * np.random.randn()
        elif noise_model == "uniform":
            e_k = np.random.uniform(-noise_level, noise_level)
        else:
            raise ValueError("noise_model must be 'gaussian' or 'uniform'.")

        y[k] = C * x[k] + e_k
        x[k + 1] = A * x[k] + B * u[k]

    return x, y


def build_siso_n1_regressor(u, y):
    u = np.asarray(u, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)

    Phi = np.column_stack([u[:-1], y[:-1]])
    Y = y[1:]

    return Phi, Y


def true_theta_siso_n1(A, B, C):
    return np.array([C * B, A], dtype=float)


# LS
def least_squares(Phi, Y):
    theta_hat, _, _, _ = np.linalg.lstsq(Phi, Y, rcond=None)
    return theta_hat.reshape(-1)


# Adaptive weights
def make_adaptive_weights(Phi, Y, delta=1e-6, power=1.0):
    theta_init = np.linalg.pinv(Phi) @ Y.reshape(-1, 1)
    theta_init = theta_init.flatten()

    weights = 1.0 / ((np.abs(theta_init) + delta) ** power)
    weights = weights / np.mean(weights)

    return weights


# CONSTRAINED weighted lasso
def weighted_lasso_constrained(Phi, Y, epsilon, weights=None, lambda_l2=0.0):
    Phi = np.asarray(Phi, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)

    T, p = Phi.shape

    if weights is None:
        weights = np.ones(p)

    weights = np.asarray(weights, dtype=float).reshape(-1)

    theta = cp.Variable(p)
    residual = Phi @ theta - Y

    weighted_l1 = cp.norm1(cp.multiply(weights, theta))

    if lambda_l2 > 0:
        objective = cp.Minimize(
            weighted_l1 + 0.5 * lambda_l2 * cp.sum_squares(theta)
        )
    else:
        objective = cp.Minimize(weighted_l1)

    constraints = [
        cp.norm(residual, 2) <= float(epsilon) * np.sqrt(T)
    ]

    prob = cp.Problem(objective, constraints)

    for solver, kwargs in [
        (
            cp.ECOS,
            dict(
                warm_start=True,
                verbose=False,
                max_iters=8000,
                reltol=1e-7,
                abstol=1e-7,
                feastol=1e-7,
            ),
        ),
        (
            cp.SCS,
            dict(
                warm_start=True,
                verbose=False,
                eps=1e-5,
                max_iters=30000,
                alpha=1.8,
                acceleration_lookback=20,
            ),
        ),
    ]:
        try:
            prob.solve(solver=solver, **kwargs)
            if theta.value is not None:
                return theta.value.reshape(-1)
        except Exception:
            continue

    raise ValueError("Constrained weighted lasso failed.")


# Fixed epsilon selection
def compute_epsilon_star(noise_level, theta_y):
    """
    Use the same epsilon for every realization.

    For e(k) ~ U[-noise_level, noise_level],

        mu = noise_level / sqrt(3),

    and

        epsilon_star
        = 1.05 * mu * sqrt(1 + ||theta_y||_2^2).

    Thus, the theoretical threshold is increased by 5%.
    """
    theta_y = np.asarray(theta_y, dtype=float).reshape(-1)

    mu = float(noise_level) / np.sqrt(3.0)

    epsilon_theoretical = mu * np.sqrt(
        1 + np.linalg.norm(theta_y, 2) ** 2
    )

    return 1.05 * epsilon_theoretical


# Feasible ellipse
def feasible_ellipse_2d(Phi, Y, epsilon, n_points=600):
    T = Phi.shape[0]

    theta_ls = least_squares(Phi, Y)
    residual = Y - Phi @ theta_ls
    residual_norm_sq = float(residual @ residual)

    eps_min = np.sqrt(residual_norm_sq / T)
    rho_sq = float(epsilon**2 * T - residual_norm_sq)

    if rho_sq <= 0:
        raise ValueError(
            f"The feasible set is empty. eps={epsilon:.6g}, eps_min={eps_min:.6g}. "
            "Increase EPSILON_MULT or set EPSILON manually."
        )

    M = Phi.T @ Phi
    eigvals, eigvecs = np.linalg.eigh(M)

    if np.min(eigvals) <= 1e-12:
        raise ValueError("Phi.T @ Phi is nearly singular. Increase N_SAMPLES.")

    M_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

    t = np.linspace(0.0, 2.0 * np.pi, n_points)
    circle = np.vstack([np.cos(t), np.sin(t)])

    ellipse = theta_ls.reshape(2, 1) + np.sqrt(rho_sq) * M_inv_sqrt @ circle

    return {
        "theta_ls": theta_ls,
        "ellipse": ellipse,
        "rho_sq": rho_sq,
        "eps_min": eps_min,
        "residual_norm": np.sqrt(residual_norm_sq),
    }


# Keep only points inside displayed feasible set
def filter_points_inside_feasible_set(Phi, Y, theta_all, epsilon, tol=1e-10):
    Phi = np.asarray(Phi, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    theta_all = np.asarray(theta_all, dtype=float)

    T = Phi.shape[0]
    bound = float(epsilon) * np.sqrt(T)

    keep = []

    for theta in theta_all:
        residual_norm = np.linalg.norm(Y - Phi @ theta, 2)
        keep.append(residual_norm <= bound + tol)

    keep = np.array(keep, dtype=bool)

    return theta_all[keep], keep


# Monte Carlo estimates and one ellipse per feasible realization
def monte_carlo_estimates(
    A,
    B,
    C,
    N_samples,
    noise_level,
    noise_model,
    num_trials,
    weight_delta,
    weight_power,
    lambda_l2,
    random_seed,
    epsilon_star,
    ellipse_points=400,
    feasibility_tolerance=1e-12,
):
    """
    Use the same fixed epsilon_star for every realization and discard
    realizations whose feasible set is empty.

    A realization is retained only if

        eps_min,omega < epsilon_star,

    where eps_min,omega is the normalized LS residual.
    """
    rng_state = np.random.get_state()
    np.random.seed(random_seed)

    theta_star = true_theta_siso_n1(A, B, C)
    epsilon_star = float(epsilon_star)

    theta_ls_kept = []
    theta_lasso_kept = []
    eps_min_kept = []
    eps_true_kept = []
    ellipses = []
    Phi_kept = []
    Y_kept = []
    kept_trial_indices = []
    discarded_trial_indices = []

    for trial in range(num_trials):
        u = np.random.randn(N_samples)

        _, y = simulate_siso_n1(
            A,
            B,
            C,
            u,
            noise_level=noise_level,
            noise_model=noise_model,
        )

        Phi, Y = build_siso_n1_regressor(u, y)
        theta_ls = least_squares(Phi, Y)
        T = Phi.shape[0]

        eps_min = np.linalg.norm(Y - Phi @ theta_ls) / np.sqrt(T)
        eps_true = np.linalg.norm(Y - Phi @ theta_star) / np.sqrt(T)

        tolerance = float(feasibility_tolerance) * max(
            1.0,
            abs(epsilon_star),
            abs(eps_min),
        )

        # At eps_min == epsilon_star, the ellipse collapses to the LS point.
        # Keep only strictly nonempty, nondegenerate ellipses.
        if epsilon_star <= eps_min + tolerance:
            discarded_trial_indices.append(trial)
            continue

        weights = make_adaptive_weights(
            Phi,
            Y,
            delta=weight_delta,
            power=weight_power,
        )

        theta_lasso = weighted_lasso_constrained(
            Phi,
            Y,
            epsilon=epsilon_star,
            weights=weights,
            lambda_l2=lambda_l2,
        )

        ellipse_result = feasible_ellipse_2d(
            Phi,
            Y,
            epsilon_star,
            n_points=ellipse_points,
        )

        theta_ls_kept.append(theta_ls)
        theta_lasso_kept.append(theta_lasso)
        eps_min_kept.append(eps_min)
        eps_true_kept.append(eps_true)
        ellipses.append(ellipse_result["ellipse"])
        Phi_kept.append(Phi)
        Y_kept.append(Y)
        kept_trial_indices.append(trial)

    np.random.set_state(rng_state)

    if not kept_trial_indices:
        raise ValueError(
            "No realization produced a nonempty ellipse with the fixed "
            f"epsilon_star={epsilon_star:.6g}."
        )

    theta_ls_all = np.asarray(theta_ls_kept, dtype=float)
    theta_lasso_all = np.asarray(theta_lasso_kept, dtype=float)
    eps_min_all = np.asarray(eps_min_kept, dtype=float)
    eps_true_all = np.asarray(eps_true_kept, dtype=float)
    epsilon_all = np.full(len(kept_trial_indices), epsilon_star, dtype=float)

    return {
        "theta_ls_all": theta_ls_all,
        "theta_lasso_all": theta_lasso_all,
        "epsilon_star": epsilon_star,
        "epsilon_all": epsilon_all,
        "eps_min_all": eps_min_all,
        "eps_true_all": eps_true_all,
        "ellipses": ellipses,
        "Phi_all": Phi_kept,
        "Y_all": Y_kept,
        "kept_trial_indices": np.asarray(kept_trial_indices, dtype=int),
        "discarded_trial_indices": np.asarray(
            discarded_trial_indices,
            dtype=int,
        ),
        "num_requested": int(num_trials),
        "num_kept": len(kept_trial_indices),
        "num_discarded": len(discarded_trial_indices),
    }


# Plot all realization-specific ellipses and estimates
def plot_all_feasible_sets_with_estimates(
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
    epsilon_all,
):
    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    # Plot one feasible ellipse for every realization.
    for i, ellipse in enumerate(ellipses):
        ax.plot(
            ellipse[0, :],
            ellipse[1, :],
            linewidth=0.8,
            alpha=0.22,
            color="tab:blue",
            label=(
                r"Realization-specific feasible sets"
                if i == 0
                else None
            ),
        )

    ax.scatter(
        theta_ls_all[:, 0],
        theta_ls_all[:, 1],
        s=30,
        alpha=0.70,
        marker="o",
        facecolors="none",
        edgecolors="black",
        linewidths=0.9,
        label=r"LS estimates",
        zorder=4,
    )

    ax.scatter(
        theta_lasso_all[:, 0],
        theta_lasso_all[:, 1],
        s=34,
        alpha=0.82,
        marker="x",
        color="tab:orange",
        linewidths=1.1,
        label=r"Weighted-lasso estimates",
        zorder=5,
    )

    ax.plot(
        theta_star[0],
        theta_star[1],
        marker="*",
        markersize=16,
        markeredgecolor="black",
        linestyle="None",
        color="tab:green",
        label=r"$\theta^\star$",
        zorder=6,
    )

    epsilon_fixed = float(epsilon_all[0])

    ax.set_xlabel(r"$\theta_u$")
    ax.set_ylabel(r"$\theta_y$")
    ax.set_title(
        rf"All {len(ellipses)} realization-specific feasible sets, "
        rf"$\varepsilon^*={epsilon_fixed:.3g}$"
    )
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_aspect("equal", adjustable="datalim")

    ax.legend(
        frameon=True,
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.20),
        ncol=2,
        borderaxespad=0.0,
    )

    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.90])

    return fig, ax


# Manual TikZ export
def save_tikz(
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
    tex_path,
):
    ellipse_blocks = []

    for i, ellipse in enumerate(ellipses):
        ellipse_coords = "\n".join(
            f"({ellipse[0, j]:.12g},{ellipse[1, j]:.12g})"
            for j in range(ellipse.shape[1])
        )

        legend_option = (
            "\n\\addlegendentry{Realization-specific feasible sets}"
            if i == 0
            else ""
        )

        ellipse_blocks.append(
            rf"""\addplot[
    steelblue,
    line width=0.45pt,
    opacity=0.22
] coordinates {{
{ellipse_coords}
}};{legend_option}
"""
        )

    all_ellipse_code = "\n".join(ellipse_blocks)

    ls_coords = "\n".join(
        f"({theta_ls_all[i, 0]:.12g},{theta_ls_all[i, 1]:.12g})"
        for i in range(theta_ls_all.shape[0])
    )

    lasso_coords = "\n".join(
        f"({theta_lasso_all[i, 0]:.12g},{theta_lasso_all[i, 1]:.12g})"
        for i in range(theta_lasso_all.shape[0])
    )

    tex = rf"""% Manually generated PGFPlots file.
% Requires:
% \usepackage{{xcolor}}
% \usepackage{{tikz}}
% \usepackage{{pgfplots}}
% \pgfplotsset{{compat=1.18}}

\begin{{tikzpicture}}

\definecolor{{steelblue}}{{RGB}}{{31,119,180}}
\definecolor{{darkorange}}{{RGB}}{{255,127,14}}
\definecolor{{forestgreen}}{{RGB}}{{44,160,44}}

\begin{{axis}}[
    width=0.88\linewidth,
    height=0.72\linewidth,
    xlabel={{$\theta_u$}},
    ylabel={{$\theta_y$}},
    axis lines=box,
    axis equal image,
    grid=major,
    legend style={{
        fill=white,
        draw=black,
        at={{(0.5,1.18)}},
        anchor=south,
        font=\scriptsize
    }},
    legend columns=2,
    legend cell align={{left}},
    tick align=outside,
    tick pos=left
]

{all_ellipse_code}

\addplot[
    only marks,
    mark=o,
    mark size=1.8pt,
    black,
    fill=white,
    opacity=0.70
] coordinates {{
{ls_coords}
}};
\addlegendentry{{LS estimates}}

\addplot[
    only marks,
    mark=x,
    mark size=2.0pt,
    darkorange,
    opacity=0.82
] coordinates {{
{lasso_coords}
}};
\addlegendentry{{Weighted-lasso estimates}}

\addplot[
    only marks,
    mark=star,
    mark size=3.8pt,
    forestgreen,
    mark options={{draw=black}}
] coordinates {{
({theta_star[0]:.12g},{theta_star[1]:.12g})
}};
\addlegendentry{{$\theta^\star$}}

\end{{axis}}
\end{{tikzpicture}}
"""

    tex_path = Path(tex_path)
    tex_path.parent.mkdir(parents=True, exist_ok=True)
    tex_path.write_text(tex)


def save_all(
    fig,
    ellipses,
    theta_star,
    theta_ls_all,
    theta_lasso_all,
):
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    pdf_path = FIG_DIR / f"{FIG_BASENAME}.pdf"
    png_path = FIG_DIR / f"{FIG_BASENAME}.png"
    tex_path = FIG_DIR / f"{FIG_BASENAME}.tex"

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")

    save_tikz(
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
        tex_path=tex_path,
    )

    print(f"Saved PDF:  {pdf_path}")
    print(f"Saved PNG:  {png_path}")
    print(f"Saved TikZ: {tex_path}")


# Main


### Main experiment

This block sets the numerical parameters, runs the experiment, and saves the corresponding figure outputs.


In [ ]:
def main():
    theta_star = true_theta_siso_n1(A, B, C)
    theta_y = np.array([theta_star[1]], dtype=float)
    epsilon_star = compute_epsilon_star(NOISE_LEVEL, theta_y)

    mc = monte_carlo_estimates(
        A=A,
        B=B,
        C=C,
        N_samples=N_SAMPLES,
        noise_level=NOISE_LEVEL,
        noise_model=NOISE_MODEL,
        num_trials=NUM_TRIALS,
        weight_delta=WEIGHT_DELTA,
        weight_power=WEIGHT_POWER,
        lambda_l2=LAMBDA_L2,
        random_seed=RANDOM_SEED,
        epsilon_star=epsilon_star,
        ellipse_points=ELLIPSE_POINTS,
    )

    theta_ls_all = mc["theta_ls_all"]
    theta_lasso_all = mc["theta_lasso_all"]
    epsilon_all = mc["epsilon_all"]
    ellipses = mc["ellipses"]

    print("theta size:", 2)
    print("theta_star =", theta_star)
    print(f"number of realizations = {NUM_TRIALS}")
    print(f"fixed epsilon_star (5% increased) = {epsilon_star:.6g}")
    print(f"realizations requested = {mc['num_requested']}")
    print(f"realizations kept = {mc['num_kept']}")
    print(f"realizations discarded = {mc['num_discarded']}")
    print(f"epsilon min  = {np.min(epsilon_all):.6g}")
    print(f"epsilon mean = {np.mean(epsilon_all):.6g}")
    print(f"epsilon max  = {np.max(epsilon_all):.6g}")
    print(
        "mean LS error     =",
        np.mean(np.linalg.norm(theta_ls_all - theta_star, axis=1)),
    )
    print(
        "mean LASSO error  =",
        np.mean(np.linalg.norm(theta_lasso_all - theta_star, axis=1)),
    )

    fig, ax = plot_all_feasible_sets_with_estimates(
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
        epsilon_all=epsilon_all,
    )

    save_all(
        fig=fig,
        ellipses=ellipses,
        theta_star=theta_star,
        theta_ls_all=theta_ls_all,
        theta_lasso_all=theta_lasso_all,
    )

    plt.show()

    return {
        "theta_star": theta_star,
        "epsilon_star": mc["epsilon_star"],
        "kept_trial_indices": mc["kept_trial_indices"],
        "discarded_trial_indices": mc["discarded_trial_indices"],
        "num_requested": mc["num_requested"],
        "num_kept": mc["num_kept"],
        "num_discarded": mc["num_discarded"],
        "theta_ls_all": theta_ls_all,
        "theta_lasso_all": theta_lasso_all,
        "epsilon_all": epsilon_all,
        "eps_min_all": mc["eps_min_all"],
        "eps_true_all": mc["eps_true_all"],
        "ellipses": ellipses,
        "Phi_all": mc["Phi_all"],
        "Y_all": mc["Y_all"],
        "fig": fig,
        "ax": ax,
    }


# User choices
RANDOM_SEED = 1

N_SAMPLES = 1000
NUM_TRIALS = 80
ELLIPSE_POINTS = 500

# Noise model:
#   "gaussian": e(k) ~ N(0, NOISE_LEVEL^2)
#   "uniform":  e(k) ~ U[-NOISE_LEVEL, NOISE_LEVEL]
NOISE_MODEL = "uniform"
NOISE_LEVEL = 0.10

# For uniform noise U[-NOISE_LEVEL, NOISE_LEVEL],
#   mu = NOISE_LEVEL / sqrt(3).
# Use the same fixed value for every realization:
#   epsilon_star = 1.05 * mu * sqrt(1 + ||theta_y||_2^2).
# Thus, the theoretical threshold is increased by 5%.
# Realizations with eps_min >= epsilon_star are discarded.
# Here theta_y = A = 0.7.

# Constrained weighted-lasso settings
WEIGHT_DELTA = 1e-6
WEIGHT_POWER = 1.0
LAMBDA_L2 = 0.0

FIG_DIR = Path("Figures")
FIG_BASENAME = "all_realization_feasible_sets_uniform_noise_5_percent"


# SISO n=1 system
A = 0.70
B = 1.00
C = 1.00


if __name__ == "__main__":
    results_uniform_feasible_siso = main()


## Figures 7–8 — Feasible-set evolution with sample size

This section follows one uniform-noise realization as the sample size changes, producing the feasible-set evolution and ellipse-area figures.

### Functions

| Function | Main inputs | Purpose |
|---|---|---|
| `true_theta_siso_n1` | `A, B, C` | Return the true observer parameter vector for the first-order SISO system. |
| `simulate_single_uniform_realization` | `A, B, C, u, noise_half_width, random_seed` | Generate one fixed input/output realization with uniform measurement noise. |
| `build_siso_n1_regressor` | `u, y, N` | Build the regression matrix and aligned output vector for the first-order observer model. |
| `least_squares` | `Phi, Y` | Compute the ordinary least-squares parameter estimate. |
| `compute_uniform_epsilon` | `noise_half_width, theta_y, inflation_factor` | Compute the residual bound for the uniform-noise experiment. |
| `feasible_ellipse_2d` | `Phi, Y, epsilon, n_points` | Construct the two-dimensional feasible ellipse from the residual constraint. |
| `analyze_single_realization` | `A, B, C, N_values, noise_half_width, input_seed, noise_seed, inflation_factor, ellipse_points` | Evaluate the feasible ellipse and its area over the requested sample sizes. |
| `plot_ellipses_vs_N` | `analysis` | Plot the feasible ellipses for the selected sample sizes. |
| `plot_area_vs_N` | `analysis` | Plot feasible-ellipse area as a function of sample size. |
| `save_tikz_ellipse_figure` | `analysis, tex_path, max_points` | Export the ellipse-evolution figure to TikZ/LaTeX. |
| `save_tikz_area_figure` | `analysis, tex_path` | Export the ellipse-area figure to TikZ/LaTeX. |
| `save_summary_csv` | `analysis, path` | Save the numerical summary of the sample-size study to CSV. |


In [ ]:
# Feasible ellipse shrinkage versus sample size N
# Single uniform-noise realization
# TikZ export: two figures
#   1) ellipses versus N
#   2) ellipse area versus N

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


# Model utilities
def true_theta_siso_n1(A, B, C):
    return np.array([C * B, A], dtype=float)


def simulate_single_uniform_realization(A, B, C, u, noise_half_width, random_seed):
    """
    Generate one fixed realization of the complete experiment.
    The returned y and e arrays are reused through prefixes for all N.
    """
    u = np.asarray(u, dtype=float).reshape(-1)
    n_max = len(u)

    rng = np.random.default_rng(random_seed)
    e = rng.uniform(-float(noise_half_width), float(noise_half_width), size=n_max)

    x = np.zeros(n_max + 1, dtype=float)
    y = np.zeros(n_max, dtype=float)

    for k in range(n_max):
        y[k] = C * x[k] + e[k]
        x[k + 1] = A * x[k] + B * u[k]

    return x, y, e


def build_siso_n1_regressor(u, y, N):
    """
    Use the first N input-output samples.
    Phi has T=N-1 rows:
        Phi[k] = [u(k), y(k)]
        Y[k]   = y(k+1)
    """
    if N < 3:
        raise ValueError("N must be at least 3.")

    u_n = np.asarray(u[:N], dtype=float)
    y_n = np.asarray(y[:N], dtype=float)

    Phi = np.column_stack([u_n[:-1], y_n[:-1]])
    Y = y_n[1:]
    return Phi, Y


def least_squares(Phi, Y):
    theta_ls, _, _, _ = np.linalg.lstsq(Phi, Y, rcond=None)
    return theta_ls.reshape(-1)


# Epsilon for uniform noise
def compute_uniform_epsilon(noise_half_width, theta_y, inflation_factor=1.05):
    """
    For e(k) ~ U[-h,h],
        mu = h/sqrt(3),
    and
        epsilon* = mu sqrt(1 + ||theta_y||_2^2).
    """
    theta_y = np.asarray(theta_y, dtype=float).reshape(-1)

    mu = float(noise_half_width) / np.sqrt(3.0)
    epsilon_star = mu * np.sqrt(1.0 + np.linalg.norm(theta_y, 2) ** 2)
    epsilon_used = float(inflation_factor) * epsilon_star
    return epsilon_star, epsilon_used


# Feasible ellipse geometry
def feasible_ellipse_2d(Phi, Y, epsilon, n_points=240):
    """
    Return the ellipse boundary and its semi-axis lengths.
    """
    Phi = np.asarray(Phi, dtype=float)
    Y = np.asarray(Y, dtype=float).reshape(-1)

    T = Phi.shape[0]
    theta_ls = least_squares(Phi, Y)

    residual = Y - Phi @ theta_ls
    residual_norm_sq = float(residual @ residual)
    eps_min = np.sqrt(residual_norm_sq / T)
    rho_sq = float(epsilon**2 * T - residual_norm_sq)

    if rho_sq <= 0.0:
        return {
            "is_nonempty": False,
            "theta_ls": theta_ls,
            "eps_min": eps_min,
            "rho_sq": rho_sq,
            "ellipse": None,
            "semi_axes": None,
            "major_radius": np.nan,
            "minor_radius": np.nan,
            "area": np.nan,
        }

    M = Phi.T @ Phi
    eigvals, eigvecs = np.linalg.eigh(M)

    if np.min(eigvals) <= 1e-12:
        return {
            "is_nonempty": False,
            "theta_ls": theta_ls,
            "eps_min": eps_min,
            "rho_sq": rho_sq,
            "ellipse": None,
            "semi_axes": None,
            "major_radius": np.nan,
            "minor_radius": np.nan,
            "area": np.nan,
        }

    semi_axes = np.sqrt(rho_sq / eigvals)

    t = np.linspace(0.0, 2.0 * np.pi, n_points)
    unit_circle = np.vstack([np.cos(t), np.sin(t)])

    ellipse = theta_ls.reshape(2, 1) + eigvecs @ np.diag(semi_axes) @ unit_circle

    return {
        "is_nonempty": True,
        "theta_ls": theta_ls,
        "eps_min": eps_min,
        "rho_sq": rho_sq,
        "ellipse": ellipse,
        "semi_axes": semi_axes,
        "major_radius": float(np.max(semi_axes)),
        "minor_radius": float(np.min(semi_axes)),
        "area": float(np.pi * np.prod(semi_axes)),
    }


# Analyze one realization for increasing N
def analyze_single_realization(
    A,
    B,
    C,
    N_values,
    noise_half_width,
    input_seed,
    noise_seed,
    inflation_factor,
    ellipse_points,
):
    N_values = np.asarray(N_values, dtype=int)

    if np.any(N_values < 3):
        raise ValueError("Every N must be at least 3.")

    if np.any(np.diff(N_values) <= 0):
        raise ValueError("N_values must be strictly increasing.")

    N_max = int(np.max(N_values))

    input_rng = np.random.default_rng(input_seed)
    u = input_rng.standard_normal(N_max)

    x, y, e = simulate_single_uniform_realization(
        A=A,
        B=B,
        C=C,
        u=u,
        noise_half_width=noise_half_width,
        random_seed=noise_seed,
    )

    theta_star = true_theta_siso_n1(A, B, C)
    theta_y = np.array([theta_star[1]], dtype=float)

    epsilon_theoretical, epsilon_used = compute_uniform_epsilon(
        noise_half_width=noise_half_width,
        theta_y=theta_y,
        inflation_factor=inflation_factor,
    )

    results = []
    for N in N_values:
        Phi, Y = build_siso_n1_regressor(u, y, int(N))
        ellipse_result = feasible_ellipse_2d(
            Phi=Phi,
            Y=Y,
            epsilon=epsilon_used,
            n_points=ellipse_points,
        )

        results.append(
            {
                "N": int(N),
                "T": int(Phi.shape[0]),
                "Phi": Phi,
                "Y": Y,
                **ellipse_result,
            }
        )

    return {
        "theta_star": theta_star,
        "epsilon_theoretical": epsilon_theoretical,
        "epsilon_used": epsilon_used,
        "inflation_factor": inflation_factor,
        "u": u,
        "x": x,
        "y": y,
        "e": e,
        "results": results,
    }


# Plot ellipses for selected N
def plot_ellipses_vs_N(analysis):
    theta_star = analysis["theta_star"]
    results = analysis["results"]

    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    valid_results = [r for r in results if r["is_nonempty"]]
    if not valid_results:
        raise ValueError("All feasible sets are empty for the selected N values.")

    for result in valid_results:
        ellipse = result["ellipse"]
        N = result["N"]

        ax.plot(
            ellipse[0, :],
            ellipse[1, :],
            linewidth=1.0,
            alpha=0.78,
            label=rf"$N={N}$",
        )

        theta_ls = result["theta_ls"]
        ax.plot(
            theta_ls[0],
            theta_ls[1],
            marker="o",
            markersize=4.0,
            markerfacecolor="none",
            markeredgecolor="black",
            linestyle="None",
        )

    ax.plot(
        theta_star[0],
        theta_star[1],
        marker="*",
        markersize=15,
        markeredgecolor="black",
        linestyle="None",
        label=r"$\theta^\star$",
        zorder=10,
    )

    ax.set_xlabel(r"$\theta_u$")
    ax.set_ylabel(r"$\theta_y$")
    ax.set_title(
        rf"Single uniform-noise realization, "
        rf"$\varepsilon={analysis['inflation_factor']:.2f}\varepsilon^*$"
    )
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(frameon=True, fontsize=8, ncol=2, loc="best")

    fig.tight_layout()
    return fig, ax


# Plot ellipse area versus N
def plot_area_vs_N(analysis):
    results = [r for r in analysis["results"] if r["is_nonempty"]]
    N = np.array([r["N"] for r in results], dtype=int)
    area = np.array([r["area"] for r in results], dtype=float)

    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    ax.plot(N, area, marker="o", linewidth=1.4, label="Ellipse area")

    ax.set_xlabel(r"Sample size $N$")
    ax.set_ylabel("Ellipse area")
    ax.set_title("Feasible-ellipse area versus sample size")
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.legend(frameon=True)

    fig.tight_layout()
    return fig, ax


# TikZ export
def save_tikz_ellipse_figure(analysis, tex_path, max_points=64):
    """
    TikZ/PGFPlots export for ellipses versus N.
    """
    results = [r for r in analysis["results"] if r["is_nonempty"]]
    theta_star = analysis["theta_star"]

    def downsample_coords(ellipse):
        pts = np.column_stack([ellipse[0, :], ellipse[1, :]])
        n = pts.shape[0]
        if n <= max_points:
            sel = pts
        else:
            idx = np.linspace(0, n - 1, max_points, dtype=int)
            idx = np.unique(idx)
            sel = pts[idx]
            if np.linalg.norm(sel[0] - sel[-1]) > 1e-12:
                sel[-1] = sel[0]
        return sel

    ellipse_blocks = []
    for res in results:
        coords = downsample_coords(res["ellipse"])
        coord_lines = "\n".join(f"({x:.7g},{y:.7g})" for x, y in coords)
        ellipse_blocks.append(
            rf"""\addplot[
    steelblue,
    line width=0.45pt,
    opacity=0.22,
    forget plot
] coordinates {{
{coord_lines}
}};"""
        )

    ls_coords = "\n".join(
        f"({r['theta_ls'][0]:.7g},{r['theta_ls'][1]:.7g})" for r in results
    )

    ellipse_code = "\n\n".join(ellipse_blocks)

    tex = rf"""% Compact PGFPlots export of ellipses versus N.

\begin{{tikzpicture}}

\definecolor{{steelblue}}{{RGB}}{{31,119,180}}
\definecolor{{forestgreen}}{{RGB}}{{44,160,44}}

\begin{{axis}}[
    width=0.88\linewidth,
    height=0.72\linewidth,
    xlabel={{$\theta_u$}},
    ylabel={{$\theta_y$}},
    axis lines=box,
    axis equal image,
    grid=major,
    legend style={{
        fill=white,
        draw=black,
        at={{(0.5,1.18)}},
        anchor=south,
        font=\scriptsize
    }},
    legend columns=2,
    legend cell align={{left}},
    tick align=outside,
    tick pos=left
]

{ellipse_code}

\addplot[
    only marks,
    mark=o,
    mark size=1.8pt,
    black,
    mark options={{draw=black,fill=white}},
    opacity=0.70,
    forget plot
] coordinates {{
{ls_coords}
}};

\addplot[
    only marks,
    mark=star,
    mark size=3.8pt,
    forestgreen,
    mark options={{draw=black,fill=forestgreen}},
    opacity=1,
    forget plot
] coordinates {{
({theta_star[0]:.7g},{theta_star[1]:.7g})
}};

% Independent legend samples.
\addlegendimage{{
    steelblue,
    line width=1.0pt
}}
\addlegendentry{{Realization-specific feasible sets}}

\addlegendimage{{
    only marks,
    mark=o,
    mark size=2.2pt,
    mark options={{draw=black,fill=white}},
    opacity=1
}}
\addlegendentry{{LS estimates}}

\addlegendimage{{
    only marks,
    mark=star,
    mark size=3.8pt,
    mark options={{draw=black,fill=forestgreen}},
    forestgreen,
    opacity=1
}}
\addlegendentry{{$\theta^\star$}}

\end{{axis}}
\end{{tikzpicture}}
"""
    tex_path = Path(tex_path)
    tex_path.parent.mkdir(parents=True, exist_ok=True)
    tex_path.write_text(tex, encoding="utf-8")


def save_tikz_area_figure(analysis, tex_path):
    """
    TikZ/PGFPlots export for area versus N: one plot only.
    """
    results = [r for r in analysis["results"] if r["is_nonempty"]]
    coords = "\n".join(
        f"({r['N']},{r['area']:.7g})" for r in results
    )

    tex = rf"""% Compact PGFPlots export of area versus N.

\begin{{tikzpicture}}

\begin{{axis}}[
    width=0.85\linewidth,
    height=0.60\linewidth,
    xlabel={{Sample size $N$}},
    ylabel={{Ellipse area}},
    axis lines=box,
    grid=major,
    legend style={{
        fill=white,
        draw=black,
        at={{(0.5,1.16)}},
        anchor=south,
        font=\scriptsize
    }},
    tick align=outside,
    tick pos=left
]

\addplot[
    thick,
    mark=o,
    mark size=1.8pt,
    black
] coordinates {{
{coords}
}};
\addlegendentry{{Ellipse area}}

\end{{axis}}
\end{{tikzpicture}}
"""
    tex_path = Path(tex_path)
    tex_path.parent.mkdir(parents=True, exist_ok=True)
    tex_path.write_text(tex, encoding="utf-8")


# Save numerical summary
def save_summary_csv(analysis, path):
    rows = []
    for r in analysis["results"]:
        rows.append(
            [
                r["N"],
                r["T"],
                int(r["is_nonempty"]),
                r["eps_min"],
                r["major_radius"],
                r["minor_radius"],
                r["area"],
                r["theta_ls"][0],
                r["theta_ls"][1],
            ]
        )

    header = (
        "N,T,is_nonempty,eps_min,major_radius,minor_radius,"
        "ellipse_area,theta_ls_u,theta_ls_y"
    )

    np.savetxt(
        path,
        np.asarray(rows, dtype=float),
        delimiter=",",
        header=header,
        comments="",
    )


# Main


### Main experiment

This block sets the numerical parameters, runs the experiment, and saves the corresponding figure outputs.


In [ ]:
def main():
    analysis = analyze_single_realization(
        A=A,
        B=B,
        C=C,
        N_values=N_VALUES,
        noise_half_width=NOISE_HALF_WIDTH,
        input_seed=INPUT_SEED,
        noise_seed=NOISE_SEED,
        inflation_factor=EPSILON_INFLATION,
        ellipse_points=ELLIPSE_POINTS,
    )

    print("Single uniform-noise realization")
    print("--------------------------------")
    print("theta_star =", analysis["theta_star"])
    print(f"theoretical epsilon_star = {analysis['epsilon_theoretical']:.8f}")
    print(
        f"used epsilon = {analysis['epsilon_used']:.8f} "
        f"= {EPSILON_INFLATION:.2f} epsilon_star"
    )
    print()

    print("N      eps_min       major radius    minor radius    area")
    print("-" * 68)
    for r in analysis["results"]:
        if r["is_nonempty"]:
            print(
                f"{r['N']:<6d} "
                f"{r['eps_min']:<13.7f} "
                f"{r['major_radius']:<15.7f} "
                f"{r['minor_radius']:<15.7f} "
                f"{r['area']:.7f}"
            )
        else:
            print(f"{r['N']:<6d} {r['eps_min']:<13.7f} EMPTY")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    fig1, ax1 = plot_ellipses_vs_N(analysis)
    fig2, ax2 = plot_area_vs_N(analysis)

    ellipse_pdf = OUTPUT_DIR / "single_uniform_realization_ellipses_vs_N.pdf"
    ellipse_png = OUTPUT_DIR / "single_uniform_realization_ellipses_vs_N.png"
    ellipse_tex = OUTPUT_DIR / "single_uniform_realization_ellipses_vs_N.tex"

    area_pdf = OUTPUT_DIR / "single_uniform_realization_area_vs_N.pdf"
    area_png = OUTPUT_DIR / "single_uniform_realization_area_vs_N.png"
    area_tex = OUTPUT_DIR / "single_uniform_realization_area_vs_N.tex"

    csv_path = OUTPUT_DIR / "single_uniform_realization_ellipse_summary.csv"

    fig1.savefig(ellipse_pdf, bbox_inches="tight")
    fig1.savefig(ellipse_png, dpi=300, bbox_inches="tight")
    fig2.savefig(area_pdf, bbox_inches="tight")
    fig2.savefig(area_png, dpi=300, bbox_inches="tight")

    save_tikz_ellipse_figure(analysis, ellipse_tex, max_points=64)
    save_tikz_area_figure(analysis, area_tex)
    save_summary_csv(analysis, csv_path)

    print()
    print(f"Saved: {ellipse_pdf}")
    print(f"Saved: {ellipse_png}")
    print(f"Saved: {ellipse_tex}")
    print(f"Saved: {area_pdf}")
    print(f"Saved: {area_png}")
    print(f"Saved: {area_tex}")
    print(f"Saved: {csv_path}")

    plt.show()

    return {
        "analysis": analysis,
        "ellipse_figure": fig1,
        "ellipse_axis": ax1,
        "area_figure": fig2,
        "area_axis": ax2,
    }


# User settings
A = 0.70
B = 1.00
C = 1.00

# Uniform noise e(k) ~ U[-0.1,0.1]
NOISE_HALF_WIDTH = 0.10

# Use epsilon = 1.05 epsilon*.
EPSILON_INFLATION = 1.05

# Increasing sample sizes. All use prefixes of the same realization.
N_VALUES = np.array([100, 200, 400, 800, 1200, 1600, 2000], dtype=int)

INPUT_SEED = 11
NOISE_SEED = 17

ELLIPSE_POINTS = 240
OUTPUT_DIR = Path("Figures/single_uniform_realization_N_study")


if __name__ == "__main__":
    results_single_uniform_realization = main()
